In [2]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F

from lib.helpers import plot_example_grid
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, DoublePullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func, quantus_double_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
loaded_df_resnet = QuantusEvaluator.load_results(f"results/quantus_resnet_20_25_model_name_resnet50")
print("num_samples:", len(loaded_df_resnet.iloc[0].iloc[0]))
summary_resnet = QuantusEvaluator.summarize_results(loaded_df_resnet)
summary_resnet

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.11±0.105,0.546±0.416,0.568±0.171,0.057±0.2,60.827±50.344,0.171±0.052,0.18±0.055,0.616±0.052,-0.08±0.355
DoublePullback,0.107±0.104,0.536±0.423,0.569±0.17,0.059±0.203,59.977±49.877,0.17±0.052,0.18±0.056,0.618±0.052,-0.077±0.355
Gradient,0.011±0.103,0.277±0.463,0.187±0.175,0.02±0.105,589.494±777.132,1.084±0.155,1.129±0.165,0.548±0.036,-0.042±0.236
GradientShap,-0.016±0.102,0.427±0.467,-0.191±0.173,0.041±0.154,1535.851±2748.603,1.136±0.19,1.374±0.293,0.613±0.049,-0.031±0.253
IntegratedGradients,-0.019±0.102,0.424±0.486,-0.208±0.183,0.043±0.161,735.057±978.154,0.942±0.196,0.983±0.209,0.612±0.049,-0.034±0.245
Saliency,0.005±0.102,0.318±0.505,-0.126±0.435,0.03±0.129,56446948±87718128,0.745±0.109,0.78±0.12,0.48±0.042,0.41±0.095
DeepLift,-0.012±0.099,0.292±0.519,-0.186±0.171,0.041±0.158,414.154±605.901,1.102±0.166,1.158±0.182,0.62±0.049,-0.036±0.251
InputXGradient,-0.016±0.099,0.312±0.513,-0.183±0.17,0.041±0.158,416.579±591.101,1.101±0.165,1.156±0.177,0.62±0.049,-0.036±0.25
Deconvolution,0.005±0.099,0.421±0.413,0.222±0.247,0.011±0.075,6968249344±5074092032,0.707±0.119,0.726±0.118,0.542±0.008,1±0


In [11]:
loaded_df_vgg = QuantusEvaluator.load_results(f"results/quantus_vgg_20_25_vgg11_bn")
print("num_samples:", len(loaded_df_vgg.iloc[0].iloc[0]))
summary_vgg = QuantusEvaluator.summarize_results(loaded_df_vgg)
summary_vgg

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.069±0.11,0.463±0.455,0.565±0.214,0.005±0.028,54.736±69.833,0.253±0.065,0.26±0.067,0.568±0.041,-0.05±0.305
DoublePullback,0.061±0.106,0.455±0.452,0.573±0.21,0.007±0.039,43.377±53.441,0.247±0.058,0.256±0.061,0.572±0.041,-0.016±0.29
Gradient,0.014±0.101,0.23±0.485,0.158±0.161,0.005±0.039,367.498±480.932,0.987±0.124,1.009±0.127,0.521±0.037,-0.043±0.269
GradientShap,-0.018±0.103,0.322±0.49,-0.201±0.167,0.014±0.078,1250.731±2906.688,0.986±0.133,1.141±0.179,0.591±0.054,-0.035±0.253
IntegratedGradients,-0.025±0.1,0.345±0.486,-0.211±0.17,0.014±0.08,639.19±800.556,0.842±0.128,0.858±0.128,0.594±0.054,-0.039±0.259
Saliency,0.009±0.104,0.418±0.457,-0.007±0.401,0.008±0.058,50356508±58405256,0.645±0.072,0.661±0.076,0.442±0.045,0.411±0.106
DeepLift,-0.02±0.103,0.214±0.534,-0.199±0.16,0.014±0.083,317.133±381.703,1.004±0.129,1.031±0.132,0.597±0.053,-0.041±0.271
InputXGradient,-0.008±0.094,0.234±0.535,-0.199±0.16,0.014±0.083,316.903±370.885,1.004±0.129,1.031±0.132,0.597±0.053,-0.041±0.271
Deconvolution,0.002±0.097,0.005±0.481,-0.051±0.14,0.005±0.03,63309988±48742628,0.784±0.163,0.793±0.164,0.493±0.005,0.998±0.001
